# Chuẩn bị thư viện

In [23]:
#!pip3 install selenium
#!mkdir -p data
from urllib.parse import urljoin
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.common.exceptions import StaleElementReferenceException, NoSuchElementException
from selenium.webdriver.chrome.options import Options
import time
import os
import json

In [24]:
# selenium setups
## https://www.tutorialspo/int.com/selenium/selenium_webdriver_chrome_webdriver_options.htm

chrome_options = webdriver.ChromeOptions()

chrome_options.add_argument('--headless') # must options for Google Colab
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')
chrome_options.add_argument("--disable-extensions")
chrome_options.add_argument("--disable-gpu")


In [25]:
MAGAZINE_NAME = "thuvienphapluat"
HOME_PAGE = "https://thuvienphapluat.vn/hoi-dap-phap-luat/quyen-dan-su"

# Thu thập thử 1 sample

## Thu thập link

In [26]:
driver = webdriver.Chrome(options=chrome_options)
# Vào trang web chính, mặc định phải chờ toàn bộ trang webload mới xong
driver.get(HOME_PAGE)

In [27]:
#Collect link hỏi đáp trong mục dân sự

articles = driver.find_elements(By.CSS_SELECTOR, "article.news-card")

results = []
for art in articles:
    try:
        link_tag = art.find_element(By.TAG_NAME, "a")
        href = link_tag.get_attribute("href").strip()
        title = link_tag.get_attribute("title").strip()

        results.append({
            "title": title,
            "href": href
        })
    except Exception as e:
        print("Lỗi đọc 1 article:", e)
    break

In [28]:
href

'https://thuvienphapluat.vn/hoi-dap-phap-luat/luat-thuc-hien-dan-chu-o-co-so-moi-nhat-la-luat-nao-138066122.html'

In [29]:
title

'Luật Thực hiện dân chủ ở cơ sở mới nhất là Luật nào?'

## Thu thập nội dung câu hỏi và context trong 1 link vừa crawl

In [30]:
def _clean_text(s: str) -> str:
    return " ".join(s.split()).strip()

def get_qas_from_tvpl(driver, article_url):
    """
    Trả về list[{"query": str, "context": str}] từ 1 trang hỏi–đáp của thuvienphapluat.vn
    """
    driver.get(article_url)
    wait = WebDriverWait(driver, 10)

    # Phần nội dung hỏi–đáp
    content = wait.until(
        EC.presence_of_element_located((By.CSS_SELECTOR, "section.news-content"))
    )

    # Duyệt theo thứ tự DOM để giữ đúng nhóm {h2 -> các p/ul/ol...}
    nodes = content.find_elements(By.XPATH, "./*")

    qas = []
    current_q = None
    buf = []

    def flush():
        nonlocal current_q, buf, qas
        # Kết thúc 1 QA khi có query hiện tại và đã gom được nội dung
        if current_q and buf:
            ctx = "\n".join([_clean_text(x) for x in buf if _clean_text(x)])
            if ctx:
                qas.append({"query": _clean_text(current_q), "context": ctx})
        current_q, buf = None, []

    for el in nodes:
        tag = el.tag_name.lower()

        # Bỏ qua các phần không phải nội dung
        if tag in {"script", "style", "noscript"}:
            continue
            
        if len(el.find_elements(By.TAG_NAME, "em")) > 0:
            continue

        if tag == "h2":
            # Gặp câu hỏi mới => chốt QA cũ (nếu có)
            flush()
            current_q = el.text.strip()  # bên trong thường có <strong>
            continue

        # Gom phần trả lời cho câu hỏi hiện tại
        if current_q:
            if tag in {"p", "blockquote"}:
                txt = el.text.strip()
                if txt:
                    buf.append(txt)
            elif tag in {"ul", "ol"}:
                # Lấy từng gạch đầu dòng
                items = [li.text.strip() for li in el.find_elements(By.TAG_NAME, "li")]
                items = [f"- {t}" for t in items if t]
                if items:
                    buf.append("\n".join(items))

    # Chốt QA cuối cùng
    flush()
    return qas


In [31]:
qas = get_qas_from_tvpl(driver, href)

In [32]:
qas

[{'query': 'Luật Thực hiện dân chủ ở cơ sở mới nhất là Luật nào?',
  'context': 'Luật Thực hiện dân chủ ở cơ sở 2022 được Quốc hội thông qua ngày 10/11/2022, trong đó quy định việc thực hiện dân chủ ở xã, phường, thị trấn, cơ quan Nhà nước, đơn vị sự nghiệp công lập và tổ chức sử dụng lao động.\nLuật Thực hiện dân chủ ở cơ sở 2022 có hiệu lực từ ngày 01/7/2023.\nTính đến tháng 10/2025, Luật Thực hiện dân chủ ở cơ sở 2022 vẫn có hiệu lực thi hành và chưa có văn bản thay thế.\nDo đó, Luật Thực hiện dân chủ ở cơ sở mới nhất là Luật Thực hiện dân chủ ở cơ sở 2022.\nVăn bản sửa đổi Luật Thực hiện dân chủ ở cơ sở 2022 gồm:\n- Luật sửa đổi Luật Mặt trận Tổ quốc Việt Nam, Luật Công đoàn, Luật Thanh niên và Luật Thực hiện dân chủ ở cơ sở 2025\n- Luật Quy hoạch đô thị và nông thôn 2024'},
 {'query': 'Công dân thực hiện dân chủ tại đâu?',
  'context': 'Tại Điều 4 Luật Thực hiện dân chủ ở cơ sở 2022 được sửa đổi bởi khoản 3 Điều 4 Luật sửa đổi Luật Mặt trận Tổ quốc Việt Nam, Luật Công đoàn, Luật T

# Chạy thật

## Thu thập URL QA

In [33]:
options = Options()
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
driver = webdriver.Chrome(options=options)

base_url = "https://thuvienphapluat.vn/hoi-dap-phap-luat/quyen-dan-su"
driver.get(base_url)
time.sleep(2)

url_list = []

# Lấy số trang cuối
last_page = driver.find_elements(By.CSS_SELECTOR, "ul.pagination li.page-item a.page-link")[-1]
last_page_num = int(last_page.get_attribute("aria-label"))
print(f"Tổng số trang: {last_page_num}")

for page in range(1, last_page_num + 1):
    page_url = f"{base_url}?page={page}"
    print("Đang crawl:", page_url)
    driver.get(page_url)
    time.sleep(2)
    
    # Lấy các bài hỏi đáp trong trang
    articles = driver.find_elements(By.CSS_SELECTOR, "article.news-card a.title-link")
    for a in articles:
        title = a.get_attribute("title").strip()
        href = a.get_attribute("href").strip()
        url_list.append((title, href))

driver.quit()

Tổng số trang: 1560
Đang crawl: https://thuvienphapluat.vn/hoi-dap-phap-luat/quyen-dan-su?page=1
Đang crawl: https://thuvienphapluat.vn/hoi-dap-phap-luat/quyen-dan-su?page=2
Đang crawl: https://thuvienphapluat.vn/hoi-dap-phap-luat/quyen-dan-su?page=3
Đang crawl: https://thuvienphapluat.vn/hoi-dap-phap-luat/quyen-dan-su?page=4
Đang crawl: https://thuvienphapluat.vn/hoi-dap-phap-luat/quyen-dan-su?page=5
Đang crawl: https://thuvienphapluat.vn/hoi-dap-phap-luat/quyen-dan-su?page=6
Đang crawl: https://thuvienphapluat.vn/hoi-dap-phap-luat/quyen-dan-su?page=7
Đang crawl: https://thuvienphapluat.vn/hoi-dap-phap-luat/quyen-dan-su?page=8
Đang crawl: https://thuvienphapluat.vn/hoi-dap-phap-luat/quyen-dan-su?page=9
Đang crawl: https://thuvienphapluat.vn/hoi-dap-phap-luat/quyen-dan-su?page=10
Đang crawl: https://thuvienphapluat.vn/hoi-dap-phap-luat/quyen-dan-su?page=11
Đang crawl: https://thuvienphapluat.vn/hoi-dap-phap-luat/quyen-dan-su?page=12
Đang crawl: https://thuvienphapluat.vn/hoi-dap-phap-l

KeyboardInterrupt: 

In [34]:
len(url_list)

1007

In [35]:
import pandas as pd

df = pd.DataFrame(url_list)
df.to_csv("url_list.csv", index=False, encoding="utf-8-sig")
df.head()

,0,1
0,Luật Thực hiện dân chủ ở cơ sở mới nhất là Luậ...,https://thuvienphapluat.vn/hoi-dap-phap-luat/l...
1,Hạn chót nộp giấy tạm hoãn nghĩa vụ quân sự 20...,https://thuvienphapluat.vn/hoi-dap-phap-luat/h...
2,Hồ sơ đăng ký tạm trú cho trẻ em gồm những giấ...,https://thuvienphapluat.vn/hoi-dap-phap-luat/h...
3,Đăng ký tạm trú cho trẻ em ở đâu?,https://thuvienphapluat.vn/hoi-dap-phap-luat/d...
4,Một nghĩa vụ được bảo đảm thực hiện bằng nhiều...,https://thuvienphapluat.vn/hoi-dap-phap-luat/m...


## Crawl query-context ở từng link

In [ ]:
all_qas = []
for idx, row in df.iterrows():
    title = str(row[0]).strip()
    url   = str(row[1]).strip()

    if not url or not url.startswith("http"):
        continue

    try:
        qas = get_qas_from_tvpl(driver, url)   
        # thêm nguồn vào từng QA để truy vết
        for qa in qas:
            all_qas.append({
                "query": qa["query"],
                "context": qa["context"]
            })
        print(f"[{idx+1}/{len(df)}] OK: {title} ({len(qas)} QA)")
    except Exception as e:
        print(f"[{idx+1}/{len(df)}] LỖI: {url} -> {e}")

    time.sleep(0.5) 

driver.quit()

# --- lưu JSON ---
out_path = "QA_law_data.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(all_qas, f, ensure_ascii=False, indent=2)

[1/1007] OK: Luật Thực hiện dân chủ ở cơ sở mới nhất là Luật nào? (3 QA)
[2/1007] OK: Hạn chót nộp giấy tạm hoãn nghĩa vụ quân sự 2026 là khi nào? (3 QA)
[3/1007] OK: Hồ sơ đăng ký tạm trú cho trẻ em gồm những giấy tờ gì? (3 QA)
[4/1007] OK: Đăng ký tạm trú cho trẻ em ở đâu? (3 QA)
[5/1007] OK: Một nghĩa vụ được bảo đảm thực hiện bằng nhiều tài sản thì phạm vi bảo đảm xác định như thế nào? (3 QA)
[6/1007] OK: Nghị định hướng dẫn Luật Cư trú mới nhất là nghị định nào? (3 QA)
[7/1007] OK: Tòa án ra quyết định tuyên bố người mất năng lực hành vi dân sự trên cơ sở nào? (3 QA)
[8/1007] OK: Nghị định hướng dẫn Luật Hộ tịch là văn bản nào? (3 QA)
[9/1007] OK: Hà Nội: Chi tiết thời gian đi nghĩa vụ quân sự và thực hiện nghĩa vụ tham gia Công an nhân dân năm 2026 như thế nào? (3 QA)
[10/1007] OK: Luật Nghĩa vụ quân sự có bao nhiêu Chương bao nhiêu Điều? (3 QA)
[11/1007] OK: Tổng hợp 10 mẫu Bản cam kết được nhiều người sử dụng nhất 2025? (3 QA)
[12/1007] OK: Bộ luật Tố tụng dân sự mới nhất 2025 